# 制御理論の基礎（第7回）: 伝達関数と時間応答 (完全修正版)

このノートブックは、これまでのエラーをすべて修正した、完全に動作するバージョンです。

## 0) 準備セル: 依存関係とユーティリティ
最初のセルで、日本語フォントのダウンロードと設定、および必要な関数の定義をすべて行います。

In [ ]:
# 0) Setup (Jupyter Lite / Pyodide)
import sys, math, io, os, types
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display
import pyodide_http
import matplotlib.font_manager as fm
from matplotlib.animation import FuncAnimation

# --- Font Setup ---
pyodide_http.patch_all()

async def setup_font():
    font_url = 'https://cdn.jsdelivr.net/npm/noto-sans-jp@2.0.0/fonts/NotoSansJP-Regular.otf'
    font_path = 'NotoSansJP-Regular.otf'
    try:
        from pyodide.http import pyfetch
        if not os.path.exists(font_path):
            print(f"Downloading font: {font_url}")
            response = await pyfetch(font_url)
            with open(font_path, "wb") as f:
                f.write(await response.bytes())
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'Noto Sans JP'
        print('Japanese font setup complete.')
    except Exception as e:
        print(f"Font setup failed: {e}, using default font.")
        plt.rcParams['font.family'] = 'DejaVu Sans'

await setup_font()

# --- Utility Functions ---
sp.init_printing(use_unicode=True)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (5, 3)
plt.rcParams['axes.grid'] = True

def safe_lambdify(var, expr, modules='numpy'):
    custom_module = {'Heaviside': lambda x: np.heaviside(x, 1)}
    return sp.lambdify(var, expr, modules=[modules, custom_module])

## 1) ラプラス変換と伝達関数の定義

In [ ]:
# 1) Symbols
t, s = sp.symbols('t s', real=True)
u = sp.Function('u')
y = sp.Function('y')
U_step = 1 / s

## 2) 比例・積分要素の時間応答

In [ ]:
# 2-1) 比例要素: G(s) = K
K = sp.symbols('K', positive=True)
G_prop = K
Y_prop_s = G_prop * U_step
y_prop_t = sp.inverse_laplace_transform(Y_prop_s, s, t)
y_prop_t

In [ ]:
# 2-2) 積分要素: G(s) = 1/s
G_int = 1 / s
Y_int_s = G_int * U_step
y_int_t = sp.inverse_laplace_transform(Y_int_s, s, t)
y_int_t

In [ ]:
# 2-3) 数値化と可視化
def step_time_series(y_expr, T=5.0, dt=0.01):
    ts = np.arange(0, T + dt, dt)
    if not y_expr.free_symbols:
        ys = np.full_like(ts, float(y_expr))
    else:
        f = safe_lambdify(t, y_expr, 'numpy')
        ys = f(ts)
    return ts, ys

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for kval in [0.5, 1.0, 2.0]:
    y_prop = y_prop_t.subs({K: kval})
    ts, ys = step_time_series(y_prop, T=3.0)
    axes[0].plot(ts, ys, label='K={}'.format(kval))
axes[0].set_title('比例要素: ステップ応答')
axes[0].set_xlabel('t [s]')
axes[0].set_ylabel('y(t)')
axes[0].legend()

ts, ys = step_time_series(y_int_t, T=3.0)
axes[1].plot(ts, ys, label='G(s)=1/s')
axes[1].set_title('積分要素: ステップ応答 (ランプ)')
axes[1].set_xlabel('t [s]')
axes[1].set_ylabel('y(t)')
axes[1].legend()
plt.tight_layout()
plt.show()